## Read The data and analyze it

In [1]:
import pandas as pd 
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")
train.columns, test.columns

(Index(['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'], dtype='object'),
 Index(['id', 'prompt', 'A', 'B', 'C', 'D', 'E'], dtype='object'))

In [2]:
# shape and value count of answer column
print("Shape of test and train dataset")
print(f'train dataset shape : {train.shape}, test datatset shape : {test.shape}')
print()
print("Value count of answer column of train dataset")
frequency_of_option = train['answer'].value_counts()
print(frequency_of_option)
print()
# Q1. # Sum of most frequently and least frequently occure option 
sum_option = frequency_of_option.iloc[0] + frequency_of_option.iloc[-1]
print("Sum of least and most frequency option")
print(sum_option)

Shape of test and train dataset
train dataset shape : (2000, 8), test datatset shape : (500, 7)

Value count of answer column of train dataset
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Sum of least and most frequency option
814


##### Distribution of options in target column is noe so much imbalaced. 

## Handling missing values

In [3]:
print("Missing data in tarin & Test")
print(f'train : \n{train.isnull().sum()} , test : \n{test.isnull().sum()}')

Missing data in tarin & Test
train : 
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64 , test : 
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
dtype: int64


##### The dataset does not contain missing values in either the training or test set. Therefore, no imputation or missing-value handling is required before preprocessing and feature extraction.

## Text Preprocessing & Vocabulary Analysis

In [4]:
import string 

# Function for cleaning row 
def clean_text(text) :
    text = text.lower()

    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )
    return text

In [5]:
cleaned_prompt = train["prompt"].apply(clean_text)

# vocabulary set 
vocabulary = set() 

for prompt in cleaned_prompt :
    vocabulary.update(prompt.split())

print("Total unique words in prrompt column")
print("Vocabulary Size:", len(vocabulary))

Total unique words in prrompt column
Vocabulary Size: 859


In [6]:
# Check for all columns prompt , A, B , C, D, E 
all_text = []

for col in ["prompt", "A", "B", "C", "D", "E"]:
    cleaned = train[col].apply(clean_text) 

    all_text.extend(cleaned)

full_vocab = set() 

for text in all_text :
    full_vocab.update(text.split())

print("vocabulary Size across the prompt and options Columns is " ,len(full_vocab))

vocabulary Size across the prompt and options Columns is  3096


In [7]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
# let's see the first row of prompt 
row = train.loc[0, "prompt"]

clean_row = clean_text(row)

tokens = clean_row.split()
filter_tokens = [ word for word in tokens if word not in ENGLISH_STOP_WORDS]

print("after removing commom words, unique word in prompt at zero index ",len(filter_tokens))
# i get 13 unique words  

after removing commom words, unique word in prompt at zero index  13


##### Now analyse the prompt and all options columns A, B , C , D , E

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer 

combined_text = (train["prompt"] + " "+
                 train["A"] + " " +
                 train["B"] + " "+
                 train["C"] + " "+
                 train["D"] + " "+
                 train["E"] )

vectorizer = TfidfVectorizer(stop_words="english")

X = vectorizer.fit_transform(combined_text)
print(X.shape)
print("Total unique words across options and promt columns are :  ",len(vectorizer.get_feature_names_out()))    

(2000, 2762)
Total unique words across options and promt columns are :   2762


##### Answer choices contain a huge amount of information.(2762 - 859 = 1903 unique words in answer choices)

In [9]:
print(train.loc[0, "prompt"])
print()
print(train.loc[0, "A"])

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.

Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.


In [10]:
# cosine similarity 
from sklearn.metrics.pairwise import cosine_similarity
prompt1 = train.loc[0,"prompt"]
option_a = train.loc[0,"A"]
option_b = train.loc[0,"B"]
option_c = train.loc[0,"C"]
option_d = train.loc[0,"D"]
option_e = train.loc[0,"E"]

prompt_vec = vectorizer.transform([prompt1])
option_a_vec = vectorizer.transform([option_a])
option_b_vec = vectorizer.transform([option_b])
option_c_vec = vectorizer.transform([option_c])
option_d_vec = vectorizer.transform([option_d])
option_e_vec = vectorizer.transform([option_e])

similarity_with_a = cosine_similarity(prompt_vec, option_a_vec)[0][0]
similarity_with_b = cosine_similarity(prompt_vec, option_b_vec)[0][0]
similarity_with_c = cosine_similarity(prompt_vec, option_c_vec)[0][0]
similarity_with_d = cosine_similarity(prompt_vec, option_d_vec)[0][0]
similarity_with_e = cosine_similarity(prompt_vec, option_e_vec)[0][0]

results = {
    "A": similarity_with_a,
    "B": similarity_with_b,
    "C": similarity_with_c,
    "D": similarity_with_d,
    "E": similarity_with_e
}

predicted = max(results, key=results.get)

print("similarity with a " ,similarity_with_a)
print("similarity with b ",similarity_with_b)
print("similarity with c ",similarity_with_c)
print("similarity with d ",similarity_with_d)
print("similarity with e ",similarity_with_e) 
print(predicted)

print(train.loc[0,'answer'])

similarity with a  0.27202429519891635
similarity with b  0.3036286931760937
similarity with c  0.5876662698435949
similarity with d  0.5385329310060211
similarity with e  0.23662770351943294
C
B


In [11]:
# cosine similarity for all rows 
correct_prediction = 0 

for idx, row in train.iterrows():
    prompt = row["prompt"]

    option_a = row["A"]
    option_b = row["B"]
    option_c = row["C"]
    option_d = row["D"]
    option_e = row["E"]

    actual_answer = row["answer"]

    prompt_vec = vectorizer.transform([prompt])

    option_a_vec = vectorizer.transform([option_a])
    option_b_vec = vectorizer.transform([option_b])
    option_c_vec = vectorizer.transform([option_c])
    option_d_vec = vectorizer.transform([option_d])
    option_e_vec = vectorizer.transform([option_e])

    scores = {
    "A": cosine_similarity(prompt_vec, option_a_vec)[0][0],
    "B": cosine_similarity(prompt_vec, option_b_vec)[0][0],
    "C": cosine_similarity(prompt_vec, option_c_vec)[0][0],
    "D": cosine_similarity(prompt_vec, option_d_vec)[0][0],
    "E": cosine_similarity(prompt_vec, option_e_vec)[0][0]
     }
    predicted_answer = max(scores, key=scores.get)

    if predicted_answer == actual_answer:
        correct_prediction += 1

accuracy = (correct_prediction / len(train)) * 100

print("Accuracy:", accuracy)

Accuracy: 13.55


##### random top 1 expected accuracy is 0.2 (20%) for 5 options. But we get 13,55 % means overlap with words of prompt and answer choices is not much. So we need to do some feature engineering to improve the accuracy.

##### TF-IDF captures lexical overlap between prompt and options but cannot understand semantics, context, or reasoning. Therefore, the option with the highest similarity is often not the correct answer.

# Map@3 Function Define

In [12]:
def map_at_3(actual, prediction) :
    if actual in prediction :
        rank = prediction.index(actual) + 1 

        return 1 / rank 
    return 0
# Test
print(map_at_3("C", ["C", "A", "B"]))
print(map_at_3("B", ["D", "B", "E"]))
print(map_at_3("A", ["D", "B", "C"]))

1.0
0.5
0


## Majority Baseline

In [13]:
## dummy prediction
scores =  []
for _,row in train.iterrows() :
    actual = row['answer'] 

    predict = ["B", "C", "A"]

    scores.append(map_at_3(actual, predict))

final_map3 = sum(scores) / len(scores)

print(final_map3)

0.42125


# TFIDF Vectorizer Pipiline

In [14]:
scores = []

for _, row in train.iterrows():

    prompt = row["prompt"]

    actual = row["answer"]

    prompt_vec = vectorizer.transform([prompt])

    similarities = {
        "A": cosine_similarity(prompt_vec,
                               vectorizer.transform([row["A"]]))[0][0],

        "B": cosine_similarity(prompt_vec,
                               vectorizer.transform([row["B"]]))[0][0],

        "C": cosine_similarity(prompt_vec,
                               vectorizer.transform([row["C"]]))[0][0],

        "D": cosine_similarity(prompt_vec,
                               vectorizer.transform([row["D"]]))[0][0],

        "E": cosine_similarity(prompt_vec,
                               vectorizer.transform([row["E"]]))[0][0]
    }

    ranked_options = sorted(
        similarities,
        key=similarities.get,
        reverse=True
    )

    top3_prediction = ranked_options[:3]

    scores.append(
        map_at_3(actual, top3_prediction)
    )

final_map3 = sum(scores) / len(scores)

print("TF-IDF MAP@3 =", final_map3)

TF-IDF MAP@3 = 0.2961666666666667


In [15]:
tfidf_top3_predictions = []

scores = []

for _, row in train.iterrows():

    prompt = row["prompt"]

    options = {
        "A": row["A"],
        "B": row["B"],
        "C": row["C"],
        "D": row["D"],
        "E": row["E"]
    }

    prompt_vec = vectorizer.transform([prompt])

    similarities = {}

    for label, text in options.items():

        option_vec = vectorizer.transform([text])

        similarities[label] = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

    ranked = sorted(
        similarities,
        key=similarities.get,
        reverse=True
    )

    top3 = ranked[:3]

    tfidf_top3_predictions.append(top3)

    scores.append(
        map_at_3(
            row["answer"],
            top3
        )
    )

tfidf_map3 = sum(scores) / len(scores)

In [17]:
import pickle

with open("tfidf_top3_predictions.pkl", "wb") as f:
    pickle.dump(tfidf_top3_predictions, f)

print("Saved TF-IDF predictions")

Saved TF-IDF predictions


### Observation :
- Dataset contains no missing values.
- Prompt vocabulary size = 859.
- Combined prompt + options vocabulary size = 2762.
- Majority class baseline MAP@3 = 0.42125.
- TF-IDF ranking baseline MAP@3 = 0.29617.
- TF-IDF relies on lexical overlap and does not capture semantic reasoning.
- More advanced transformer-based models are expected to significantly outperform TF-IDF.